In [1]:
"""# LLM as a Judge Code — HuggingFace Qwen3-0.6B Backend"""

import json
import os
import re
import time
import logging
import traceback
from tqdm import tqdm

from transformers import AutoModelForCausalLM, AutoTokenizer

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('political_evaluation.log'),
        logging.StreamHandler()
    ]
)


class Qwen3Manager:
    """
    Manages interaction with a local HuggingFace Qwen3-0.6B model for evaluation tasks.
    Loads model and tokenizer once; generates via transformers pipeline.
    """

    # Token ID for </think> used by Qwen3 to separate thinking from final output
    THINK_END_TOKEN_ID = 151668

    def __init__(self, model_name="Qwen/Qwen3-0.6B", max_new_tokens=8192, enable_thinking=True):
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens
        self.enable_thinking = enable_thinking
        self.total_calls = 0

        logging.info(f"Loading Qwen3 tokenizer from {self.model_name} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)

        logging.info(f"Loading Qwen3 model from {self.model_name} ...")
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            torch_dtype="auto",
            device_map="auto"
        )

        logging.info(f"Qwen3 Manager initialized with model: {self.model_name}")

    def generate_content(self, prompt, timeout=300):
        """
        Send a prompt to Qwen3-0.6B via HuggingFace transformers and return
        (thinking_content, final_content, full_decoded_text).

        Uses the official Qwen3 chat template with enable_thinking support.
        Parses the </think> boundary using token ID 151668 as per official docs.
        """
        try:
            messages = [
                {
                    "role": "system",
                    "content": (
                        "You are an expert evaluator for political meme analysis, "
                        "specializing in Bangladeshi political context. Follow the "
                        "output format instructions exactly."
                    ),
                },
                {"role": "user", "content": prompt},
            ]

            # Apply Qwen3 chat template
            text = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=self.enable_thinking,
            )
            model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

            # Generate
            generated_ids = self.model.generate(
                **model_inputs,
                max_new_tokens=self.max_new_tokens,
            )

            # Strip the input tokens to get only the new output
            output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

            # Parse thinking content vs final content using </think> token boundary
            try:
                # rindex: find last occurrence of 151668 (</think>)
                index = len(output_ids) - output_ids[::-1].index(self.THINK_END_TOKEN_ID)
            except ValueError:
                index = 0

            thinking_content = self.tokenizer.decode(
                output_ids[:index], skip_special_tokens=True
            ).strip("\n")
            final_content = self.tokenizer.decode(
                output_ids[index:], skip_special_tokens=True
            ).strip("\n")

            self.total_calls += 1

            return thinking_content, final_content

        except Exception as e:
            logging.error(f"Qwen3 generate_content error: {e}\n{traceback.format_exc()}")
            raise

    def get_usage_stats(self):
        """Get usage statistics."""
        return {
            "model": self.model_name,
            "total_calls": self.total_calls,
        }


class PoliticalMemeEvaluator:
    """
    Qwen3-0.6B evaluator for political meme analysis.
    Uses <<END_THINK>> marker to separate reasoning from final decision.
    MCQ questions are evaluated by direct comparison; open-ended questions use LLM-as-judge.
    """

    THINK_MARKER = "<<END_THINK>>"

    # Few-shot examples for political meme evaluation
    EXAMPLE_EVALUATION = """
EXAMPLE:
Model Answer: This image most likely relates to the political context of Bangladesh, given the mention of "BANGLADESH" at the top.
Ground Truth: Bangladesh
Analysis: The model answer explicitly identifies Bangladesh as the political context, which matches the ground truth. The additional explanation about how it reached this conclusion does not change the core answer.
<<END_THINK>>
{"decision":"YES", "confidence":0.95, "short_reason":"Both identify Bangladesh as the political context"}
DECISION: YES

EXAMPLE:
Model Answer: The image does not show a clear political bias favoring a particular party, ideology, or individual.
Ground Truth: Yes, the image shows strong political bias against former Prime Minister Sheikh Hasina and her party, the Awami League.
Analysis: The model answer claims no political bias, while the ground truth identifies strong bias against Sheikh Hasina and the Awami League. These are fundamentally contradictory assessments.
<<END_THINK>>
{"decision":"NO", "confidence":0.95, "short_reason":"Model says no bias, but ground truth identifies clear anti-Hasina/Awami League bias"}
DECISION: NO

EXAMPLE:
Model Answer: Yes, the image references political events in Bangladesh. 'Nirbachon' means 'election' in Bengali. This likely refers to the political discourse surrounding the 12th Bangladeshi general election held on January 7, 2024.
Ground Truth: Yes, this image references the contemporary political landscape of Bangladesh following the events of August 5, 2024. It mocks the repetitiveness of a party's rhetoric demanding immediate elections.
Analysis: Both confirm the image references political events. However, the model links it to the January 2024 election, while the ground truth links it to post-August 2024 events. The timeline and context differ significantly.
<<END_THINK>>
{"decision":"NO", "confidence":0.85, "short_reason":"Wrong timeline - model says Jan 2024 election, ground truth says post-Aug 2024 events"}
DECISION: NO

EXAMPLE:
Model Answer: The punchline lies in the reinterpretation of 'NNN' from 'No Nut November' to 'Nirbachon, Nirbachon, Nirbachon' (Election, Election, Election). It humorously captures the exhausting political discourse around elections.
Ground Truth: The punchline lies in the satirical reinterpretation of 'NNN'. The meme redefines it as Nirbachon, Nirbachon, Nirbachon. Since August 2024, Mirza Fakhrul has consistently demanded elections. This meme captures that repetitiveness.
Analysis: Both answers correctly identify the NNN → Nirbachon wordplay as the core punchline. The model captures the humor of repetitive election talk. However, the model misses the specific target (Mirza Fakhrul / BNP) and the post-August 2024 political context. The core punchline mechanism is correctly identified but key political specifics are missing.
<<END_THINK>>
{"decision":"NO", "confidence":0.70, "short_reason":"Core punchline mechanism correct but misses specific political target (Mirza Fakhrul/BNP) and correct timeline"}
DECISION: NO
"""

    def __init__(self, api_manager: Qwen3Manager, seed=42):
        self.api_manager = api_manager
        self.seed = seed
        self.model_name = api_manager.model_name

        logging.info(f"Political meme evaluator initialized with model: {self.model_name}")
        print(f"Political meme evaluator initialized with model: {self.model_name}")

    def generate_response(self, prompt):
        """
        Generate response using Qwen3-0.6B and split at THINK_MARKER.
        Qwen3 natively produces its own <think>...</think> block which is
        already parsed by Qwen3Manager. Here we further split the final
        content at our <<END_THINK>> evaluation marker.
        Returns: (thinking_text, final_text, full_response_text)
        """
        try:
            # Qwen3Manager returns (qwen_thinking, qwen_final_content)
            qwen_thinking, qwen_content = self.api_manager.generate_content(prompt, timeout=300)
            qwen_content = qwen_content.strip()

            # The evaluation prompt asks the model to output <<END_THINK>>
            # as its own reasoning/decision separator within the final content.
            # Split on that marker if present.
            marker = self.THINK_MARKER
            idx = qwen_content.rfind(marker)
            if idx >= 0:
                eval_thinking = qwen_content[:idx].strip()
                final = qwen_content[idx + len(marker):].strip()
                full = qwen_content
                return eval_thinking, final, full

            # If marker not found, treat entire content as final text
            return "", qwen_content, qwen_content

        except Exception as e:
            logging.error(f"Error generating response with Qwen3: {e}", exc_info=True)
            return "", f"Error: {str(e)}", ""


def _extract_json_block(text):
    """
    Find the last valid JSON object in text using improved pattern matching.
    Returns (json_obj, json_text) or (None, None)
    """
    if not text:
        return None, None

    # First try: regex pattern for JSON-like structures
    json_pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'
    matches = re.findall(json_pattern, text)

    for match in reversed(matches):
        try:
            obj = json.loads(match)
            return obj, match
        except:
            continue

    # Fallback: greedy approach
    last_open = text.rfind('{')
    if last_open == -1:
        return None, None

    candidate = text[last_open:]
    brace_count = 0

    for i, char in enumerate(candidate):
        if char == '{':
            brace_count += 1
        elif char == '}':
            brace_count -= 1
            if brace_count == 0:
                try:
                    maybe = candidate[:i+1]
                    obj = json.loads(maybe)
                    return obj, maybe
                except:
                    continue

    return None, None


def extract_decision_and_confidence(final_text):
    """
    Extract a decision (YES/NO) and optional confidence from the final_text.
    Returns: (decision_str_or_none, confidence_or_none, short_reason_or_none)
    """
    if not final_text:
        return None, None, None

    # 1) Try JSON block first
    obj, obj_text = _extract_json_block(final_text)
    if obj is not None:
        decision = None
        conf = None
        short = None

        for key in ("decision", "Decision", "DECISION"):
            if key in obj:
                decision_val = str(obj[key]).strip().upper()
                if decision_val in ("YES", "NO", "TRUE", "FALSE", "1", "0"):
                    decision = "YES" if decision_val in ("YES", "TRUE", "1") else "NO"
                break

        for key in ("confidence", "conf", "confidence_score"):
            if key in obj:
                try:
                    conf_val = float(obj[key])
                    if 0.0 <= conf_val <= 1.0:
                        conf = conf_val
                    elif 0.0 <= conf_val <= 100.0:
                        conf = conf_val / 100.0
                except:
                    conf = None
                break

        for key in ("short_reason", "reason", "short_reasoning"):
            if key in obj:
                short = str(obj[key])
                break

        if decision in ("YES", "NO"):
            return decision.lower(), conf, short

    # 2) Check final lines for DECISION: YES/NO
    lines = [ln.strip() for ln in final_text.strip().splitlines() if ln.strip()]
    if lines:
        last_line = lines[-1].upper().rstrip('.')
        decision_patterns = [
            "DECISION: YES", "DECISION YES", "YES", "DECISION: YES.",
            "DECISION: NO", "DECISION NO", "NO", "DECISION: NO."
        ]

        for pattern in decision_patterns:
            if last_line == pattern:
                return 'yes' if 'YES' in pattern else 'no', None, None

    # 3) Heuristic matching for political context
    tail = " ".join(lines[-3:]).upper() if lines else ""
    conclusion_indicators = ["THEREFORE", "THUS", "CONCLUSION", "FINAL", "RESULT", "OVERALL"]

    has_conclusion_context = any(indicator in tail for indicator in conclusion_indicators)

    if has_conclusion_context:
        positive_phrases = [
            "SEMANTICALLY EQUIVALENT", "SAME MEANING", "CAPTURES THE SAME",
            "CORRECTLY IDENTIFIES", "MATCHES THE GROUND TRUTH", "ALIGNS WITH"
        ]
        negative_phrases = [
            "NOT EQUIVALENT", "DOES NOT MATCH", "FAILS TO CAPTURE",
            "CONTRADICTS", "MISSES THE KEY", "INCORRECT", "FUNDAMENTALLY DIFFERENT"
        ]

        if any(phrase in tail for phrase in positive_phrases):
            return 'yes', None, None
        if any(phrase in tail for phrase in negative_phrases):
            return 'no', None, None

    return None, None, None


def evaluate_mcq(model_answer, correct_answer):
    """
    Evaluate MCQ questions by direct letter comparison.
    Returns: dict with decision, confidence, short_reason
    """
    model_clean = str(model_answer).strip().upper()
    correct_clean = str(correct_answer).strip().upper()

    is_correct = model_clean == correct_clean

    return {
        'thinking': '',
        'final_text': f"MCQ comparison: model='{model_clean}' vs correct='{correct_clean}'",
        'full_decoded': '',
        'decision': 'yes' if is_correct else 'no',
        'confidence': 1.0,
        'short_reason': f"{'Match' if is_correct else 'Mismatch'}: model={model_clean}, correct={correct_clean}",
    }


def evaluate_open_ended(evaluator, model_answer, correct_answer, question_text, image_id, question_id, max_retries=3):
    """
    Evaluate open-ended questions using Qwen3-0.6B as a semantic judge.
    Returns: dict with thinking, final_text, decision, confidence, short_reason
    """
    model_answer_text = str(model_answer) if model_answer else "No answer provided"
    correct_answer_text = str(correct_answer) if correct_answer else "No answer provided"

    if not model_answer_text.strip() or not correct_answer_text.strip():
        return {
            'thinking': '',
            'final_text': 'Missing essential data',
            'full_decoded': '',
            'decision': 'no',
            'confidence': None,
            'short_reason': 'Missing essential data',
        }

    prompt = f"""You are an expert evaluator for political meme analysis, specializing in Bangladeshi political context. Your task is to determine if the model's answer is semantically equivalent to the ground truth answer.

EVALUATION CRITERIA:

1) FACTUAL ACCURACY: The model answer must identify the same key facts, political entities (parties, leaders, institutions), events, and timelines as the ground truth. Getting the wrong political figure, party, or event timeline is a significant error.

2) POLITICAL CONTEXT: The model must demonstrate understanding of the correct political context. For Bangladeshi political memes, this includes knowledge of:
   - Key political figures (e.g., Sheikh Hasina, Mirza Fakhrul, BNP, Awami League, etc.)
   - Major political events (e.g., 2024 mass uprising, elections, political crises)
   - Political dynamics (party rivalries, alliances, public sentiment)

3) SENTIMENT & BIAS ALIGNMENT: If the ground truth identifies a specific political bias, sentiment, or target, the model must capture the same direction. Saying "no bias" when there is clear bias, or identifying the wrong target, is incorrect.

4) CORE MEANING: For humor/punchline explanations, the model must capture the same core comedic mechanism and political commentary. Identifying the right joke structure but wrong political target or context is NOT sufficient.

5) COMPLETENESS: The model answer does not need to be word-for-word identical, but it must capture the essential claims of the ground truth. Additional correct details are acceptable; missing key information is not.

6) TOLERANCE: Minor differences in wording, phrasing, or additional context that don't change the core meaning are acceptable. The model may provide more detail than the ground truth as long as the core facts align.

7) STRICTNESS ON KEY ELEMENTS: Be strict about:
   - Correct identification of political figures and parties
   - Correct event timelines
   - Correct identification of political bias direction
   - Correct understanding of satirical targets

{PoliticalMemeEvaluator.EXAMPLE_EVALUATION}

EVALUATION PROCESS:
Step 1: Identify what the question is asking (country identification, event reference, target, bias, humor, etc.)
Step 2: Extract the key claims from both the model answer and ground truth.
Step 3: Compare each key claim for factual alignment.
Step 4: Assess whether the model captures the same political understanding.
Step 5: Make final decision — YES if the core meaning and key facts align, NO if there are significant factual errors, wrong targets, or contradictory assessments.

OUTPUT REQUIREMENTS (follow exactly):
1) Give a concise step-by-step analysis showing your reasoning.
2) THEN insert the THINKING marker on its own line exactly:
{PoliticalMemeEvaluator.THINK_MARKER}
3) After the marker, provide the final machine-friendly JSON block like:
   {{"decision":"YES" or "NO", "confidence":0.0-1.0, "short_reason":"..."}}
4) Finally end the entire response with a single final line (no trailing text):
   DECISION: YES
   OR
   DECISION: NO

Now evaluate the following:

IMAGE: {image_id}
QUESTION: {question_text}

MODEL ANSWER:
{model_answer_text}

GROUND TRUTH:
{correct_answer_text}
"""

    last_result = None

    for attempt in range(1, max_retries + 1):
        thinking, final_text, full = evaluator.generate_response(prompt)
        decision, confidence, short_reason = extract_decision_and_confidence(final_text)

        result = {
            'thinking': thinking,
            'final_text': final_text,
            'full_decoded': full,
            'decision': decision,
            'confidence': confidence,
            'short_reason': short_reason,
        }
        last_result = result

        if decision in ('yes', 'no'):
            if confidence is not None:
                logging.info(f"{image_id} {question_id}: Got JSON decision with confidence {confidence}")
            return result

        logging.warning(f"Attempt {attempt}/{max_retries} failed to get decision for {image_id} {question_id}")
        if attempt < max_retries:
            time.sleep(0.1)

    # Default conservatively to 'no'
    logging.error(f"All {max_retries} attempts failed for {image_id} {question_id}; defaulting to 'no'")
    if last_result is None:
        return {
            'thinking': '',
            'final_text': 'All attempts failed - no output',
            'full_decoded': '',
            'decision': 'no',
            'confidence': None,
            'short_reason': 'Evaluation failed',
        }

    last_result['decision'] = 'no'
    if not last_result.get('short_reason'):
        last_result['short_reason'] = 'Failed to get clear decision'
    return last_result


def save_json_atomic(obj, path):
    """Save JSON file atomically to prevent corruption."""
    tmp = path + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)


def calculate_confidence_stats(results_list):
    """Calculate statistics for confidence scores from a flat list of question results."""
    confidences = [r.get('confidence') for r in results_list if r.get('confidence') is not None]
    if not confidences:
        return {"count": 0}

    return {
        "count": len(confidences),
        "mean": round(sum(confidences) / len(confidences), 4),
        "min": round(min(confidences), 4),
        "max": round(max(confidences), 4),
        "median": round(sorted(confidences)[len(confidences) // 2], 4)
    }


def main():
    """Main function to execute the political meme evaluation process."""

    # Configuration
    JSON_FILE_PATH = '/content/en_zs_results_20260425_155427.json'
    OUTPUT_FILE_PATH = '/content/gpt-oss_political_evaluation_results_english.json'

    MAX_RETRIES = 3

    # Initialize Qwen3 manager (local HuggingFace model — no API keys needed)
    qwen3_manager = Qwen3Manager(
        model_name="Qwen/Qwen3-0.6B",
        max_new_tokens=8192,
        enable_thinking=True
    )

    evaluator = PoliticalMemeEvaluator(
        api_manager=qwen3_manager,
        seed=42
    )

    # Load input data
    try:
        with open(JSON_FILE_PATH, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
        logging.info(f"Loaded {len(json_data)} images from JSON file")
        print(f"Loaded {len(json_data)} images from JSON file")
    except Exception as e:
        logging.error(f"Failed to load JSON input: {e}")
        print(f"Error loading JSON: {e}")
        return

    # Tracking variables
    image_evaluations = []
    all_question_results = []  # flat list for stats

    # Per-question-id tracking
    per_question_correct = {}   # Q1..Q7 -> count correct
    per_question_total = {}     # Q1..Q7 -> count total

    # Per-type tracking
    mcq_correct = 0
    mcq_total = 0
    open_ended_correct = 0
    open_ended_total = 0

    total_questions = 0
    total_correct = 0
    total_images = len(json_data)

    # Process each image
    for img_idx, image_entry in enumerate(tqdm(json_data, desc='Evaluating political memes')):
    # for img_idx, image_entry in enumerate(tqdm(json_data[:2], desc='Evaluating political memes')):
        try:
            image_id = image_entry.get('image_id', f'IMAGE_{img_idx+1:03d}')
            status = image_entry.get('status', 'unknown')
            questions = image_entry.get('questions', [])

            if status != 'success' or not questions:
                logging.warning(f"Skipping {image_id}: status={status}, questions={len(questions)}")
                image_evaluations.append({
                    'image_id': image_id,
                    'status': status,
                    'questions': [],
                    'image_accuracy': 0.0,
                    'note': 'Skipped - no successful results'
                })
                continue

            image_question_results = []
            image_correct = 0

            for q in questions:
                question_id = q.get('question_id', 'unknown')
                question_type = q.get('question_type', 'open_ended')
                question_text = q.get('question_text', '')
                model_answer = q.get('model_answer', '')
                correct_answer = q.get('correct_answer', '')
                options = q.get('options', None)

                # Initialize per-question tracking
                if question_id not in per_question_correct:
                    per_question_correct[question_id] = 0
                    per_question_total[question_id] = 0

                # Evaluate based on question type
                if question_type == 'mcq':
                    result = evaluate_mcq(model_answer, correct_answer)
                    mcq_total += 1
                    if result['decision'] == 'yes':
                        mcq_correct += 1
                else:
                    result = evaluate_open_ended(
                        evaluator, model_answer, correct_answer,
                        question_text, image_id, question_id,
                        max_retries=MAX_RETRIES
                    )
                    open_ended_total += 1
                    if result['decision'] == 'yes':
                        open_ended_correct += 1

                # Build question result entry
                q_result = {
                    'question_id': question_id,
                    'question_type': question_type,
                    'question_text': question_text,
                    'model_answer': model_answer,
                    'correct_answer': correct_answer,
                    'decision': result['decision'],
                    'confidence': result.get('confidence'),
                    'short_reason': result.get('short_reason'),
                    'thinking': result.get('thinking', ''),
                    'final_text': result.get('final_text', ''),
                }
                if options:
                    q_result['options'] = options

                image_question_results.append(q_result)
                all_question_results.append(q_result)

                # Update counters
                total_questions += 1
                per_question_total[question_id] += 1

                if result['decision'] == 'yes':
                    total_correct += 1
                    image_correct += 1
                    per_question_correct[question_id] += 1

                # Print progress
                decision_str = result['decision'].upper() if result['decision'] else 'UNKNOWN'
                conf_str = f" (conf: {result['confidence']:.2f})" if result.get('confidence') else ""
                print(f"  {image_id} {question_id} [{question_type}]: {decision_str}{conf_str}")

            # Image-level accuracy
            image_acc = (image_correct / len(questions)) * 100 if questions else 0.0

            image_evaluations.append({
                'image_id': image_id,
                'status': status,
                'questions': image_question_results,
                'image_correct': image_correct,
                'image_total': len(questions),
                'image_accuracy': round(image_acc, 2),
            })

            logging.info(f"{image_id}: {image_correct}/{len(questions)} correct ({image_acc:.1f}%)")

            # Intermediate save every 10 images
            if (img_idx + 1) % 10 == 0:
                overall_acc = (total_correct / total_questions) * 100 if total_questions else 0
                logging.info(f"Progress: {img_idx+1}/{total_images} images, "
                           f"Overall: {total_correct}/{total_questions} ({overall_acc:.2f}%)")
                intermediate_file = OUTPUT_FILE_PATH.replace('.json', f'_checkpoint_{img_idx+1}.json')
                save_json_atomic(image_evaluations, intermediate_file)

        except Exception as e:
            logging.error(f"Error processing {image_entry.get('image_id', img_idx)}: {e}", exc_info=True)
            image_evaluations.append({
                'image_id': image_entry.get('image_id', f'IMAGE_{img_idx+1:03d}'),
                'status': 'error',
                'questions': [],
                'image_accuracy': 0.0,
                'note': f'Error during evaluation: {str(e)}'
            })
            continue

    # Calculate final statistics
    overall_accuracy = (total_correct / total_questions) * 100 if total_questions else 0
    mcq_accuracy = (mcq_correct / mcq_total) * 100 if mcq_total else 0
    open_ended_accuracy = (open_ended_correct / open_ended_total) * 100 if open_ended_total else 0
    confidence_stats = calculate_confidence_stats(all_question_results)

    # Per-question accuracy
    per_question_accuracy = {}
    for qid in sorted(per_question_correct.keys()):
        total = per_question_total[qid]
        correct = per_question_correct[qid]
        per_question_accuracy[qid] = {
            'correct': correct,
            'total': total,
            'accuracy': round((correct / total) * 100, 2) if total else 0.0
        }

    # Prepare final results
    final_results = {
        'evaluation_metadata': {
            'model_used': evaluator.model_name,
            'total_images': total_images,
            'total_questions': total_questions,
            'total_correct': total_correct,
            'overall_accuracy_percentage': round(overall_accuracy, 2),
            'mcq_stats': {
                'total': mcq_total,
                'correct': mcq_correct,
                'accuracy_percentage': round(mcq_accuracy, 2)
            },
            'open_ended_stats': {
                'total': open_ended_total,
                'correct': open_ended_correct,
                'accuracy_percentage': round(open_ended_accuracy, 2)
            },
            'per_question_accuracy': per_question_accuracy,
            'confidence_statistics': confidence_stats,
            'evaluation_timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
            'settings': {
                'max_retries': MAX_RETRIES,
                'seed': evaluator.seed,
            }
        },
        'image_evaluations': image_evaluations,
    }

    # Save final results
    save_json_atomic(final_results, OUTPUT_FILE_PATH)

    # Print summary
    print('\n' + '='*80)
    print('POLITICAL MEME EVALUATION COMPLETED!')
    print('='*80)
    print(f"Model Used: {evaluator.model_name}")
    print(f"Total Images: {total_images}")
    print(f"Total Questions: {total_questions}")
    print(f"Total Correct: {total_correct}")
    print(f"Overall Accuracy: {overall_accuracy:.2f}%")
    print(f"\nMCQ Accuracy: {mcq_correct}/{mcq_total} ({mcq_accuracy:.2f}%)")
    print(f"Open-Ended Accuracy: {open_ended_correct}/{open_ended_total} ({open_ended_accuracy:.2f}%)")
    print(f"\nPer-Question Accuracy:")
    for qid, stats in per_question_accuracy.items():
        print(f"  {qid}: {stats['correct']}/{stats['total']} ({stats['accuracy']:.2f}%)")
    if confidence_stats.get('count', 0) > 0:
        print(f"\nConfidence Stats (open-ended):")
        print(f"  Average: {confidence_stats['mean']:.3f}")
        print(f"  Range: {confidence_stats['min']:.3f} - {confidence_stats['max']:.3f}")
    print(f"\nResults saved to: {OUTPUT_FILE_PATH}")
    print('='*80)

    # Final logging
    logging.info('Political meme evaluation completed!')
    logging.info(f"Final Results: {total_correct}/{total_questions} correct ({overall_accuracy:.2f}%)")
    logging.info(f"MCQ: {mcq_correct}/{mcq_total} ({mcq_accuracy:.2f}%)")
    logging.info(f"Open-ended: {open_ended_correct}/{open_ended_total} ({open_ended_accuracy:.2f}%)")
    logging.info(f"Results saved to: {OUTPUT_FILE_PATH}")


if __name__ == '__main__':
    main()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

... (9 outputs left out)

Evaluating political memes:  67%|██████▋   | 2/3 [03:13<01:34, 94.87s/it] 

  IMAGE_002 Q7 [open_ended]: YES (conf: 0.95)
  IMAGE_003 Q1 [open_ended]: YES (conf: 0.95)
  IMAGE_003 Q2 [open_ended]: NO
  IMAGE_003 Q3 [mcq]: YES (conf: 1.00)
  IMAGE_003 Q4 [mcq]: NO (conf: 1.00)
  IMAGE_003 Q5 [mcq]: YES (conf: 1.00)
  IMAGE_003 Q6 [open_ended]: NO


Evaluating political memes: 100%|██████████| 3/3 [04:24<00:00, 88.31s/it]

  IMAGE_003 Q7 [open_ended]: YES

POLITICAL MEME EVALUATION COMPLETED!
Model Used: Qwen/Qwen3-0.6B
Total Images: 3
Total Questions: 21
Total Correct: 11
Overall Accuracy: 52.38%

MCQ Accuracy: 5/9 (55.56%)
Open-Ended Accuracy: 6/12 (50.00%)

Per-Question Accuracy:
  Q1: 2/3 (66.67%)
  Q2: 1/3 (33.33%)
  Q3: 2/3 (66.67%)
  Q4: 1/3 (33.33%)
  Q5: 2/3 (66.67%)
  Q6: 0/3 (0.00%)
  Q7: 3/3 (100.00%)

Confidence Stats (open-ended):
  Average: 0.980
  Range: 0.950 - 1.000

Results saved to: /content/gpt-oss_political_evaluation_results_english.json
